# Crypto Price Forecasting

## Project Overview

Forecasts a **daily cryptocurrency price** (USD) over a 14-day horizon. Synthetic data spans 2 years with high volatility, bubble/crash cycles, and weekend trading patterns.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily crypto prices, predict the next 14 days. Crypto price forecasting illustrates the extreme difficulty of predicting speculative assets with regime changes and non-stationary behavior.

## Why This Project Matters

Cryptocurrency markets operate 24/7 with extreme volatility. Despite billions in trading volume, price prediction remains extremely difficult. This project teaches realistic expectations, proper evaluation, and why risk management matters more than point forecasts for speculative assets.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily crypto price
- High volatility (3-5x equity markets)
- Bubble and crash cycles
- Mild weekend volume effects
- Non-stationary with structural breaks

| Column | Description |
|--------|-------------|
| `date` | Date |
| `price_usd` | Daily crypto price (USD) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'price_usd'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

# High-volatility random walk with regime shifts
log_price = np.zeros(N_POINTS)
log_price[0] = np.log(30000)  # starting price

sigma = 0.03  # ~48% annualized vol
for i in range(1, N_POINTS):
    # Regime-dependent drift
    if 200 < i < 350:  # bubble
        drift = 0.003
        sigma_t = 0.04
    elif 350 < i < 450:  # crash
        drift = -0.004
        sigma_t = 0.05
    else:
        drift = 0.0005
        sigma_t = 0.03
    log_price[i] = log_price[i-1] + drift + sigma_t * np.random.normal()

# Weekend effects (slightly lower volume = slightly more noise)
for i in range(N_POINTS):
    if dates[i].dayofweek >= 5:
        log_price[i] += np.random.normal(0, 0.005)

price_usd = np.exp(log_price).round(2)

df = pd.DataFrame({'date': dates, 'price_usd': price_usd})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,price_usd
0,2022-01-01,29883.48
1,2022-01-02,30495.47
2,2022-01-03,30354.69
3,2022-01-04,30965.74
4,2022-01-05,32429.63


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('price_usd Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("price_usd Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

price_usd Statistics:
count      730.000000
mean     41985.768370
std      14748.621278
min      21144.730000
25%      28694.062500
50%      39855.725000
75%      50675.502500
max      89901.180000
Name: price_usd, dtype: float64

CV: 0.351
Skewness: 0.621


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:     1985.6 | RMSE:     2349.4 | MAPE:  5.13%
Seasonal Naive (7)             | MAE:     1949.7 | RMSE:     2115.7 | MAPE:  4.95%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                          Adjusted R-Squared   R-Squared          RMSE  Time Taken
Model                                                                             
KernelRidge                     12161.866827 -934.451294  42135.157624    0.056109
MLPRegressor                    10631.664933 -816.743456  39395.133051    0.299264
LinearSVR                       10424.792617 -800.830201  39009.936005    0.010009
GaussianProcessRegressor          849.941015  -64.303155  11132.715603    0.079807
DummyRegressor                     71.642654   -4.434050   3211.410786    0.006575
ExtraTreeRegressor                 68.033281   -4.156406   3128.294245    0.009480
ElasticNetCV                       39.406496   -1.954346   2367.907312    0.055286
ElasticNet                         37.491547   -1.807042   2308.120599    0.009154
TweedieRegressor                   33.913315   -1.531793   2192.038475    0.017631
KNeighborsRegressor                27.077501   -1.005962   1951.17

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: lgbm
FLAML (lgbm)                   | MAE:     1315.4 | RMSE:     1657.8 | MAPE:  3.33%
FLAML Test (lgbm)              | MAE:      718.8 | RMSE:     1010.1 | MAPE:  1.85%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:     1204.9 | RMSE:     1256.6 | MAPE:  3.05%
SF AutoETS                     | MAE:     1204.1 | RMSE:     1256.4 | MAPE:  3.04%
SF SeasonalNaive               | MAE:     1214.1 | RMSE:     1454.4 | MAPE:  3.03%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
             model         MAE        RMSE     MAPE
 FLAML Test (lgbm)  718.786495 1010.077298 1.847960
        SF AutoETS 1204.097487 1256.372778 3.044339
      SF AutoARIMA 1204.889286 1256.559101 3.046767
  SF SeasonalNaive 1214.122143 1454.427582 3.031048
      FLAML (lgbm) 1315.397794 1657.762221 3.329045
Seasonal Naive (7) 1949.713571 2115.672867 4.950675
Naive (Last Value) 1985.589286 2349.357305 5.127850

Top 3:
            model         MAE        RMSE     MAPE
FLAML Test (lgbm)  718.786495 1010.077298 1.847960
       SF AutoETS 1204.097487 1256.372778 3.044339
     SF AutoARIMA 1204.889286 1256.559101 3.046767


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -526.73, Std: 861.87


## Interpretation and Business Insight

- Crypto prices exhibit extreme volatility compared to traditional assets
- Bubble/crash cycles create non-stationary behavior
- The naive forecast is surprisingly competitive in efficient-ish markets
- Weekend trading shows slightly different behavior
- Regime detection is crucial — models trained in calm markets fail in crashes

## Limitations

1. Synthetic — real crypto depends on sentiment, regulation, whale activity
2. No on-chain metrics (hash rate, transaction volume, wallet activity)
3. No order book or exchange-specific data
4. Simplified regime structure — real crypto has more complex dynamics
5. No cross-crypto correlations (BTC dominance effects)

## How to Improve This Project

1. Add on-chain features (active addresses, exchange flows)
2. Include social sentiment (Twitter, Reddit, Fear & Greed index)
3. Model with regime-switching (Markov switching)
4. Use order flow data for short-term prediction
5. Apply Chronos-Bolt for foundation-model forecasting

## Production Considerations

- Crypto forecasting should focus on risk, not prediction
- Real-time monitoring with automated circuit breakers
- Portfolio weight adjustment based on regime detection
- Integration with exchange APIs for execution

## Common Mistakes

1. Expecting accurate point forecasts on crypto
2. Training on bull market data and deploying in a crash
3. Ignoring transaction costs and slippage
4. Using backtests without walk-forward validation
5. Not accounting for regime changes in model evaluation

## Mini Challenge / Exercises

1. Compute daily returns and compare volatility to S&P 500
2. Identify the bubble and crash phases from returns
3. Build a simple regime detector (rolling vol threshold)
4. Compare directional accuracy across different regimes

## Final Summary / Key Takeaways

1. Crypto is the hardest forecasting target due to extreme volatility
2. Regime changes make stationarity assumptions invalid
3. Risk management is far more important than point prediction
4. The naive forecast is a surprisingly strong baseline
5. On-chain and sentiment data are key for real applications

In [18]:
import json
metrics = {
    'project': 'Crypto Price Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Crypto Price Forecasting — Complete ===")

Metrics saved.

=== Crypto Price Forecasting — Complete ===
